## Example 1: Provider Discovery and Exploration

In [ ]:
import xyzservices
import xyzservices.providers as providers

# Example 1: Basic provider discovery and exploration
print("Basic provider discovery and exploration:")

# Package information
print(f"xyzservices version: {xyzservices.version}")
print(f"Package provides access to XYZ tile service definitions")

# Get all available providers
all_providers = providers.flatten()
print(f"\nTotal providers available: {len(all_providers)}")

# Explore provider categories by examining names
categories = {}
for name in all_providers.keys():
    base_name = name.split('.')[0]
    if base_name not in categories:
        categories[base_name] = []
    categories[base_name].append(name)

print(f"Provider categories: {len(categories)}")

# Show top categories by provider count
sorted_categories = sorted(categories.items(), key=lambda x: len(x[1]), reverse=True)
print(f"\nTop provider categories:")
for category, provider_list in sorted_categories[:8]:
    print(f"  {category:20}: {len(provider_list):3d} variants")

# Access a popular provider
osm_provider = providers.query_name('OpenStreetMap.Mapnik')
print(f"\nOpenStreetMap.Mapnik provider:")
print(f"  Name: {osm_provider['name']}")
print(f"  URL: {osm_provider['url']}")
print(f"  Max zoom: {osm_provider['max_zoom']}")
print(f"  Attribution: {osm_provider['attribution'][:80]}...")

# Show provider structure
print(f"  Provider type: {type(osm_provider)}")
print(f"  Available fields: {list(osm_provider.keys())}")

print("Basic provider exploration completed")

## Example 2: Provider Filtering and Search

In [ ]:
import xyzservices.providers as providers
from xyzservices import TileProvider
from urllib.parse import urlparse

# Example 2: Provider filtering and search functionality
print("Provider filtering and search functionality:")

# Get all providers for analysis
all_providers = providers.flatten()
print(f"Analyzing {len(all_providers)} providers")

# Filter providers by protocol
https_providers = {}
http_providers = {}

for name, provider in all_providers.items():
    url = provider.get('url', '')
    if url.startswith('https://'):
        https_providers[name] = provider
    elif url.startswith('http://'):
        http_providers[name] = provider

print(f"\nProtocol distribution:")
print(f"  HTTPS providers: {len(https_providers)}")
print(f"  HTTP providers:  {len(http_providers)}")
print(f"  Other/unknown:   {len(all_providers) - len(https_providers) - len(http_providers)}")

# Filter by zoom level capabilities
zoom_analysis = {'high': [], 'medium': [], 'low': []}

for name, provider in all_providers.items():
    max_zoom = provider.get('max_zoom', 0)
    if max_zoom >= 20:
        zoom_analysis['high'].append(name)
    elif max_zoom >= 15:
        zoom_analysis['medium'].append(name)
    else:
        zoom_analysis['low'].append(name)

print(f"\nZoom level capabilities:")
print(f"  High zoom (≥20):   {len(zoom_analysis['high'])} providers")
print(f"  Medium zoom (15-19): {len(zoom_analysis['medium'])} providers")
print(f"  Lower zoom (<15):    {len(zoom_analysis['low'])} providers")

# Find satellite/imagery providers
imagery_keywords = ['satellite', 'imagery', 'aerial', 'hybrid']
satellite_providers = []

for name, provider in all_providers.items():
    name_lower = name.lower()
    if any(keyword in name_lower for keyword in imagery_keywords):
        satellite_providers.append(name)

print(f"\nSatellite/Imagery providers:")
print(f"  Found {len(satellite_providers)} imagery-related providers")
if satellite_providers:
    print(f"  Examples: {satellite_providers[:5]}")

# Analyze domains and service providers
domain_analysis = {}
for name, provider in all_providers.items():
    url = provider.get('url', '')
    if url:
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.replace('{s}.', '')  # Remove subdomain placeholder
            if domain not in domain_analysis:
                domain_analysis[domain] = []
            domain_analysis[domain].append(name)
        except:
            continue

print(f"\nTop service domains:")
sorted_domains = sorted(domain_analysis.items(), key=lambda x: len(x[1]), reverse=True)
for domain, provider_names in sorted_domains[:10]:
    print(f"  {domain:30}: {len(provider_names):2d} providers")

# Search for providers by name pattern
def search_providers(pattern):
    """Search providers by name pattern"""
    matches = []
    pattern_lower = pattern.lower()
    for name, provider in all_providers.items():
        if pattern_lower in name.lower():
            matches.append((name, provider))
    return matches

# Example searches
search_terms = ['OpenStreetMap', 'Google', 'Stamen', 'CartoDB']
print(f"\nProvider search results:")
for term in search_terms:
    matches = search_providers(term)
    print(f"  '{term}': {len(matches)} matches")
    if matches:
        print(f"    Examples: {[m[0] for m in matches[:3]]}")

print("Provider filtering and search completed")

## Example 3: Custom Providers and URL Templates

In [ ]:
import xyzservices.providers as providers
from xyzservices import TileProvider

# Example 3: Custom provider creation and URL template manipulation
print("Custom provider creation and URL template manipulation:")

# Create custom tile providers
custom_providers = {}

# Example 1: Custom OpenStreetMap-style provider
custom_providers['CustomOSM'] = TileProvider({
    'name': 'CustomOSM',
    'url': 'https://custom-tiles.example.com/{z}/{x}/{y}.png',
    'max_zoom': 18,
    'attribution': 'Custom OSM tiles © Example Corp',
    'html_attribution': '<a href="https://example.com">Custom tiles</a>'
})

# Example 2: Custom satellite provider
custom_providers['CustomSat'] = TileProvider({
    'name': 'CustomSatellite',
    'url': 'https://satellite.example.com/tiles/{z}/{x}/{y}.jpg',
    'max_zoom': 22,
    'attribution': 'Satellite imagery © Example Satellite Co',
    'bounds': [[-85, -180], [85, 180]]  # Global coverage
})

# Example 3: Custom themed provider with API key
custom_providers['CustomAPI'] = TileProvider({
    'name': 'CustomAPIProvider',
    'url': 'https://api.tiles.example.com/{z}/{x}/{y}.png?key={accessToken}',
    'max_zoom': 20,
    'attribution': 'Map data © Example API',
    'accessToken': 'your-api-key-here'
})

print(f"Created {len(custom_providers)} custom providers:")
for name, provider in custom_providers.items():
    print(f"  {name}: {provider['url']}")
    print(f"    Max zoom: {provider['max_zoom']}")
    print(f"    Attribution: {provider['attribution']}")
    print()

# Demonstrate URL template manipulation
print("URL template manipulation examples:")

def build_tile_url(provider, z, x, y, **kwargs):
    """Build a tile URL from provider template and coordinates"""
    # TileProvider.build_url fills {x}/{y}/{z} AND every other placeholder the provider carries a value for
    # (here the {accessToken}); formatting the template by hand first raised KeyError: 'accessToken'.
    return provider.build_url(x=x, y=y, z=z, **kwargs)

# Test URL building with different providers
test_coordinates = [(10, 512, 256), (15, 16384, 8192), (5, 16, 8)]

print("Generated tile URLs:")
for z, x, y in test_coordinates:
    print(f"\nZoom {z}, Tile ({x}, {y}):")
    
    for name, provider in custom_providers.items():
        try:
            tile_url = build_tile_url(provider, z, x, y)
            print(f"  {name:15}: {tile_url}")
            assert f"/{z}/{x}/{y}." in tile_url and "{" not in tile_url, tile_url
        except Exception as e:
            print(f"  {name:15}: Error - {e}")
            raise

assert build_tile_url(custom_providers['CustomAPI'], 10, 512, 256) == \
    "https://api.tiles.example.com/10/512/256.png?key=your-api-key-here"

# Demonstrate provider validation
print(f"\nProvider validation:")

def validate_provider(provider):
    """Validate a tile provider configuration"""
    required_fields = ['name', 'url']
    issues = []
    
    # Check required fields
    for field in required_fields:
        if field not in provider:
            issues.append(f"Missing required field: {field}")
    
    # Validate URL template
    if 'url' in provider:
        url = provider['url']
        required_placeholders = ['{z}', '{x}', '{y}']
        missing_placeholders = [p for p in required_placeholders if p not in url]
        if missing_placeholders:
            issues.append(f"Missing URL placeholders: {missing_placeholders}")
    
    # Check zoom levels
    if 'max_zoom' in provider:
        max_zoom = provider['max_zoom']
        if not isinstance(max_zoom, int) or max_zoom < 0 or max_zoom > 30:
            issues.append(f"Invalid max_zoom: {max_zoom} (should be 0-30)")
    
    # Check attribution
    if 'attribution' not in provider:
        issues.append("Missing attribution field")
    
    return issues

# Validate custom providers
print("Validation results:")
for name, provider in custom_providers.items():
    issues = validate_provider(provider)
    status = "✓ Valid" if not issues else f"✗ {len(issues)} issues"
    print(f"  {name:20}: {status}")
    for issue in issues:
        print(f"    - {issue}")

# Compare with existing provider
print(f"\nComparing with existing provider:")
osm_original = providers.query_name('OpenStreetMap.Mapnik')
custom_osm = custom_providers['CustomOSM']

comparison_fields = ['name', 'url', 'max_zoom', 'attribution']
print("Field comparison:")
for field in comparison_fields:
    orig_val = osm_original.get(field, 'N/A')
    custom_val = custom_osm.get(field, 'N/A')
    print(f"  {field:12}: Original: {str(orig_val)[:40]}")
    print(f"  {field:12}:   Custom: {str(custom_val)[:40]}")
    print()

print("Custom provider creation and manipulation completed")

## Example 4: Tile Coordinates and Analysis

In [ ]:
import xyzservices.providers as providers
from xyzservices import TileProvider
import re

# Example 4: Provider analysis and tile coordinate calculations
print("Provider analysis and tile coordinate calculations:")

# Analyze provider characteristics
all_providers = providers.flatten()

def analyze_provider_features():
    """Analyze features across all providers"""
    features = {
        'total': len(all_providers),
        'with_max_zoom': 0,
        'with_bounds': 0, 
        'with_html_attribution': 0,
        'with_access_token': 0,
        'with_subdomains': 0,
        'zoom_levels': [],
        'url_patterns': {}
    }
    
    for name, provider in all_providers.items():
        # Count features
        if 'max_zoom' in provider:
            features['with_max_zoom'] += 1
            features['zoom_levels'].append(provider['max_zoom'])
        
        if 'bounds' in provider:
            features['with_bounds'] += 1
            
        if 'html_attribution' in provider:
            features['with_html_attribution'] += 1
            
        if 'accessToken' in provider:
            features['with_access_token'] += 1
            
        # Check for subdomain patterns
        url = provider.get('url', '')
        if '{s}' in url:
            features['with_subdomains'] += 1
        
        # Analyze URL patterns
        if url:
            # Extract file extension
            ext_match = re.search(r'\.(\w+)(?:\?|$)', url)
            if ext_match:
                ext = ext_match.group(1)
                features['url_patterns'][ext] = features['url_patterns'].get(ext, 0) + 1
    
    return features

# Perform analysis
analysis = analyze_provider_features()

print("Provider feature analysis:")
print(f"  Total providers:           {analysis['total']}")
print(f"  With max_zoom:             {analysis['with_max_zoom']} ({analysis['with_max_zoom']/analysis['total']*100:.1f}%)")
print(f"  With bounds:               {analysis['with_bounds']} ({analysis['with_bounds']/analysis['total']*100:.1f}%)")
print(f"  With HTML attribution:     {analysis['with_html_attribution']} ({analysis['with_html_attribution']/analysis['total']*100:.1f}%)")
print(f"  With access token:         {analysis['with_access_token']} ({analysis['with_access_token']/analysis['total']*100:.1f}%)")
print(f"  With subdomains:           {analysis['with_subdomains']} ({analysis['with_subdomains']/analysis['total']*100:.1f}%)")

# Zoom level statistics
if analysis['zoom_levels']:
    zoom_levels = analysis['zoom_levels']
    print(f"\nZoom level statistics:")
    print(f"  Min zoom level:            {min(zoom_levels)}")
    print(f"  Max zoom level:            {max(zoom_levels)}")
    print(f"  Average zoom level:        {sum(zoom_levels)/len(zoom_levels):.1f}")
    
    # Zoom level distribution
    zoom_dist = {}
    for zoom in zoom_levels:
        zoom_dist[zoom] = zoom_dist.get(zoom, 0) + 1
    
    print(f"  Most common zoom levels:")
    for zoom, count in sorted(zoom_dist.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"    Level {zoom:2d}: {count:3d} providers")

# File format analysis
if analysis['url_patterns']:
    print(f"\nTile format distribution:")
    for ext, count in sorted(analysis['url_patterns'].items(), key=lambda x: x[1], reverse=True):
        print(f"  .{ext:4}: {count:3d} providers ({count/analysis['total']*100:.1f}%)")

# Tile coordinate calculation utilities
def deg2tile(lat_deg, lon_deg, zoom):
    """Convert lat/lon coordinates to tile coordinates"""
    import math
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    x = int((lon_deg + 180.0) / 360.0 * n)
    y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return (x, y)

def tile2deg(x, y, zoom):
    """Convert tile coordinates to lat/lon coordinates"""
    import math
    n = 2.0 ** zoom
    lon_deg = x / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * y / n)))
    lat_deg = math.degrees(lat_rad)
    return (lat_deg, lon_deg)

# Test coordinate calculations
print(f"\nTile coordinate calculations:")

# Famous locations for testing
locations = [
    ("New York City", 40.7128, -74.0060),
    ("London", 51.5074, -0.1278),
    ("Tokyo", 35.6762, 139.6503),
    ("Sydney", -33.8688, 151.2093)
]

zoom_levels = [10, 15, 18]

print("Location to tile coordinate conversion:")
for location_name, lat, lon in locations:
    print(f"\n{location_name} ({lat:.4f}, {lon:.4f}):")
    
    for zoom in zoom_levels:
        x, y = deg2tile(lat, lon, zoom)
        # Convert back to verify
        lat_back, lon_back = tile2deg(x, y, zoom)
        
        print(f"  Zoom {zoom:2d}: Tile ({x:5d}, {y:5d}) -> ({lat_back:.4f}, {lon_back:.4f})")

# Generate sample tile URLs for different providers
print(f"\nSample tile URLs for New York City (zoom 15):")
ny_lat, ny_lon = 40.7128, -74.0060
ny_x, ny_y = deg2tile(ny_lat, ny_lon, 15)

# Stamen's tiles moved to Stadia Maps, so xyzservices now lists them as Stadia.Stamen*
sample_providers = ['OpenStreetMap.Mapnik', 'CartoDB.Positron', 'Stadia.StamenTerrain']
for provider_name in sample_providers:
    provider = providers.query_name(provider_name)
    
    # build_url fills the provider's own placeholders ({s} subdomain, {variant}, {r}, {ext}) as well as z/x/y
    sample_url = provider.build_url(x=ny_x, y=ny_y, z=15)
    print(f"  {provider_name:20}: {sample_url}")
    assert f"/15/{ny_x}/{ny_y}" in sample_url and "{" not in sample_url, sample_url

# Demonstrate bounding box calculations
def get_tile_bbox(x, y, zoom):
    """Get bounding box for a tile"""
    top_left = tile2deg(x, y, zoom)
    bottom_right = tile2deg(x + 1, y + 1, zoom)
    return {
        'north': top_left[0],
        'south': bottom_right[0], 
        'east': bottom_right[1],
        'west': top_left[1]
    }

print(f"\nTile bounding boxes:")
test_tile_x, test_tile_y, test_zoom = ny_x, ny_y, 15
bbox = get_tile_bbox(test_tile_x, test_tile_y, test_zoom)

print(f"Tile ({test_tile_x}, {test_tile_y}) at zoom {test_zoom}:")
print(f"  North: {bbox['north']:8.4f}°")
print(f"  South: {bbox['south']:8.4f}°")
print(f"  East:  {bbox['east']:8.4f}°")
print(f"  West:  {bbox['west']:8.4f}°")
print(f"  Width: {bbox['east'] - bbox['west']:8.4f}°")
print(f"  Height: {bbox['north'] - bbox['south']:8.4f}°")

print("Provider analysis and tile calculations completed")

## Example 5: Production Mapping Workflows

In [ ]:
import xyzservices.providers as providers
from xyzservices import TileProvider
import json
import time

# Example 5: Production mapping workflows and provider management
print("Production mapping workflows and provider management:")

class TileProviderManager:
    """Production-ready tile provider management system"""
    
    def __init__(self):
        self.providers = {}
        self.provider_stats = {}
        self.fallback_chain = []
        self.load_builtin_providers()
    
    def load_builtin_providers(self):
        """Load built-in xyzservices providers"""
        builtin_providers = providers.flatten()
        
        # Filter to reliable, commonly-used providers
        recommended_providers = [
            'OpenStreetMap.Mapnik',
            'CartoDB.Positron',
            'CartoDB.DarkMatter',
            'Stamen.Terrain',
            'Stamen.Watercolor'
        ]
        
        for name in recommended_providers:
            if name in builtin_providers:
                self.providers[name] = builtin_providers[name]
                self.provider_stats[name] = {
                    'requests': 0,
                    'errors': 0,
                    'last_used': None
                }
        
        print(f"Loaded {len(self.providers)} built-in providers")
    
    def add_custom_provider(self, name, config):
        """Add a custom provider to the manager"""
        # an invalid config (e.g. no `url`) raises AttributeError from TileProvider (measured) and must fail the cell
        provider = TileProvider(config)
        self.providers[name] = provider
        self.provider_stats[name] = {
            'requests': 0,
            'errors': 0,
            'last_used': None
        }
        print(f"Added custom provider: {name}")
        return True
    
    def get_provider(self, name):
        """Get a provider by name with usage tracking"""
        if name not in self.providers:
            raise ValueError(f"Provider '{name}' not found")
        
        # Update statistics
        self.provider_stats[name]['requests'] += 1
        self.provider_stats[name]['last_used'] = time.time()
        
        return self.providers[name]
    
    def build_tile_url(self, provider_name, z, x, y, **kwargs):
        """Build tile URL with error handling and fallbacks"""
        try:
            provider = self.get_provider(provider_name)
            url_template = provider['url']
            
            # Handle subdomain selection
            if '{s}' in url_template:
                # Simple subdomain rotation based on tile coordinates
                subdomains = kwargs.get('subdomains', ['a', 'b', 'c'])
                subdomain = subdomains[(x + y) % len(subdomains)]
                url_template = url_template.replace('{s}', subdomain)
            
            # Build URL
            url = url_template.format(z=z, x=x, y=y, **kwargs)
            
            # Handle access tokens
            if 'accessToken' in provider and '{accessToken}' in url:
                url = url.replace('{accessToken}', provider['accessToken'])
            
            return url, None
            
        except Exception as e:
            self.provider_stats[provider_name]['errors'] += 1
            return None, str(e)
    
    def get_provider_recommendations(self, use_case='general'):
        """Get provider recommendations based on use case"""
        recommendations = {
            'general': ['OpenStreetMap.Mapnik', 'CartoDB.Positron'],
            'dark_theme': ['CartoDB.DarkMatter'],
            'terrain': ['Stamen.Terrain'],
            'artistic': ['Stamen.Watercolor'],
            'minimal': ['CartoDB.Positron']
        }
        
        return recommendations.get(use_case, recommendations['general'])
    
    def setup_fallback_chain(self, primary_providers):
        """Setup fallback chain for reliability"""
        self.fallback_chain = primary_providers.copy()
        print(f"Fallback chain configured: {' -> '.join(self.fallback_chain)}")
    
    def get_tile_url_with_fallback(self, z, x, y, **kwargs):
        """Get tile URL with automatic fallback support"""
        for provider_name in self.fallback_chain:
            if provider_name in self.providers:
                url, error = self.build_tile_url(provider_name, z, x, y, **kwargs)
                if url:
                    return url, provider_name
                print(f"Provider {provider_name} failed: {error}")
        
        return None, "All providers in fallback chain failed"
    
    def generate_provider_report(self):
        """Generate usage and performance report"""
        report = {
            'total_providers': len(self.providers),
            'total_requests': sum(stats['requests'] for stats in self.provider_stats.values()),
            'total_errors': sum(stats['errors'] for stats in self.provider_stats.values()),
            'providers': []
        }
        
        for name, stats in self.provider_stats.items():
            provider = self.providers[name]
            
            provider_report = {
                'name': name,
                'requests': stats['requests'],
                'errors': stats['errors'],
                'error_rate': stats['errors'] / stats['requests'] * 100 if stats['requests'] > 0 else 0,
                'max_zoom': provider.get('max_zoom', 'N/A'),
                'last_used': time.ctime(stats['last_used']) if stats['last_used'] else 'Never'
            }
            report['providers'].append(provider_report)
        
        # Sort by usage
        report['providers'].sort(key=lambda x: x['requests'], reverse=True)
        return report

# Create and configure provider manager
manager = TileProviderManager()

# Add custom providers for specific use cases
custom_providers = {
    'CustomBasemap': {
        'name': 'CustomBasemap',
        'url': 'https://tiles.example.com/basemap/{z}/{x}/{y}.png',
        'max_zoom': 18,
        'attribution': 'Custom basemap tiles'
    },
    'CustomSatellite': {
        'name': 'CustomSatellite', 
        'url': 'https://satellite.example.com/{z}/{x}/{y}.jpg',
        'max_zoom': 20,
        'attribution': 'Custom satellite imagery'
    }
}

print(f"\nAdding custom providers:")
for name, config in custom_providers.items():
    success = manager.add_custom_provider(name, config)
    # the registered provider fills the tile coordinates into its own URL template
    assert success and manager.providers[name].build_url(z=1, x=2, y=3) == config['url'].format(z=1, x=2, y=3), name
    print(f"  {name}: ✓ Success")

# Setup fallback chain
fallback_providers = ['OpenStreetMap.Mapnik', 'CartoDB.Positron', 'CustomBasemap']
manager.setup_fallback_chain(fallback_providers)

# Demonstrate tile URL generation for different scenarios
print(f"\nTile URL generation scenarios:")

# Test coordinates (New York City area)
test_coords = [(10, 301, 384), (15, 9654, 12308), (18, 77238, 98464)]

for z, x, y in test_coords:
    print(f"\nZoom {z}, Tile ({x}, {y}):")
    
    # Try with fallback
    url, used_provider = manager.get_tile_url_with_fallback(z, x, y)
    if url:
        # The chain's head is the built-in OpenStreetMap.Mapnik; falling back past it means a built-in provider's URL broke
        assert used_provider == 'OpenStreetMap.Mapnik', f"fell back to {used_provider} for tile {(z, x, y)}"
        print(f"  Success: {used_provider}")
        print(f"  URL: {url}")
    else:
        print(f"  Failed: {used_provider}")
        raise AssertionError(f"every provider in the fallback chain failed for tile {(z, x, y)}")

# Test different use cases
print(f"\nProvider recommendations by use case:")
use_cases = ['general', 'dark_theme', 'terrain', 'artistic', 'minimal']

for use_case in use_cases:
    recommendations = manager.get_provider_recommendations(use_case)
    print(f"  {use_case:12}: {recommendations}")

# Generate and display provider usage report
print(f"\nProvider usage report:")
report = manager.generate_provider_report()

print(f"Overall statistics:")
print(f"  Total providers:   {report['total_providers']}")
print(f"  Total requests:    {report['total_requests']}")
print(f"  Total errors:      {report['total_errors']}")
print(f"  Overall error rate: {report['total_errors']/report['total_requests']*100:.1f}%" if report['total_requests'] > 0 else "  Overall error rate: 0.0%")

print(f"\nProvider details:")
for provider in report['providers'][:10]:  # Show top 10
    print(f"  {provider['name']:20}: {provider['requests']:3d} requests, "
          f"{provider['error_rate']:5.1f}% errors, "
          f"max_zoom={provider['max_zoom']}")

# Demonstrate tile caching strategy
print(f"\nTile caching strategy recommendations:")
print("- Cache tiles locally to reduce API requests")
print("- Implement cache expiration based on provider updates")
print("- Use provider fallback chains for high availability")
print("- Monitor provider performance and adjust usage")
print("- Respect provider rate limits and terms of service")

# Configuration export for deployment
print(f"\nDeployment configuration:")
config_export = {
    'providers': {name: dict(provider) for name, provider in manager.providers.items()},
    'fallback_chain': manager.fallback_chain,
    'recommended_use_cases': {
        use_case: manager.get_provider_recommendations(use_case)
        for use_case in ['general', 'dark_theme', 'terrain']
    }
}

print(f"Configuration contains:")
print(f"  - {len(config_export['providers'])} configured providers")
print(f"  - {len(config_export['fallback_chain'])} providers in fallback chain")
print(f"  - {len(config_export['recommended_use_cases'])} use case configurations")

print(f"\nProduction features demonstrated:")
print("- Provider lifecycle management with custom and built-in providers")
print("- Automatic fallback chains for high availability")
print("- Usage tracking and performance monitoring")
print("- Use case-specific provider recommendations")
print("- Tile URL generation with error handling")
print("- Production deployment configuration export")